In [1]:
import tensorflow as tf
import keras
import os

2026-02-10 14:16:21.282182: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 14:16:21.288480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770732981.295962 2770722 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770732981.298286 2770722 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-10 14:16:21.306101: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
gpus = tf.config.list_physical_devices(device_type='GPU')
print(gpus)
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.set_visible_devices(gpus[1], 'GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
  except RuntimeError as e:
    print(e)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2 Physical GPUs, 1 Logical GPUs


I0000 00:00:1770732983.463638 2770722 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14300 MB memory:  -> device: 1, name: NVIDIA RTX A4000, pci bus id: 0000:06:00.0, compute capability: 8.6


In [3]:
(training_images, training_labels) , \
(validation_images, validation_labels) = \
tf.keras.datasets.cifar10.load_data()

In [4]:
def preprocess_image_input(input_images):
  return input_images.astype('float32')

train_X = preprocess_image_input(training_images)
valid_X = preprocess_image_input(validation_images)

In [5]:
@keras.saving.register_keras_serializable()
class ResizeLayer(tf.keras.layers.Layer):
  def __init__(self, size, method='nearest', **kwargs):
    super(ResizeLayer, self).__init__(**kwargs)
    self.size = size
    self.method = method

  def call(self, inputs):
    return tf.image.resize(inputs, size=self.size, method=self.method)

'''
Feature Extraction is performed by EfficientNetB3 pretrained on imagenet weights. 
Input size is 224 x 224.
'''
def feature_extractor(inputs):
  feature_extractor = tf.keras.applications.EfficientNetB3(input_shape=(224, 224, 3),
                                               include_top=False,
                                               weights='imagenet')(inputs)
  return feature_extractor


'''
Defines final dense layers and subsequent softmax layer for classification.
'''
def classifier(inputs):
  x = tf.keras.layers.GlobalAveragePooling2D()(inputs)
  x = tf.keras.layers.Dropout(0.2)(x)
  x = tf.keras.layers.Flatten()(x)
  #x = tf.keras.layers.Dense(512, activation="relu")(x)
  #x = tf.keras.layers.Dropout(0.5)(x)
  x = tf.keras.layers.Dense(128, activation="relu")(x)
  x = tf.keras.layers.Dropout(0.5)(x)
  x = tf.keras.layers.Dense(10, activation="softmax", name="classification")(x)
  return x

'''
Since input image size is (32 x 32), first upsample the image by factor of (7x7) to transform it to (224 x 224)
Connect the feature extraction and "classifier" layers to build the model.
'''
def final_model(inputs):
  resize = ResizeLayer(size=(inputs.shape[1] * 7, inputs.shape[2] * 7))(inputs)
  resnet_feature_extractor = feature_extractor(resize)
  classification_output = classifier(resnet_feature_extractor)
  return classification_output

'''
Define the model and compile it. 
Use Stochastic Gradient Descent as the optimizer.
Use Sparse Categorical CrossEntropy as the loss function.
'''
def define_compile_model():
  inputs = tf.keras.layers.Input(shape=(32,32,3))
  classification_output = final_model(inputs) 
  model = tf.keras.Model(inputs=inputs, outputs = classification_output)
 
  model.compile(optimizer='SGD', 
                loss='sparse_categorical_crossentropy',
                metrics = ['accuracy'])
  
  return model

model = define_compile_model()

model.summary()

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resize_layer (ResizeLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       196,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification (Dense)          │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,981,561 (41.89 MB)

 Trainable params: 10,894,258 (41.56 MB)

 Non-trainable params: 87,303 (341.03 KB)

In [6]:
# Traint the model (only for Cifar10 dataset)
EPOCHS = 3

history = model.fit(train_X, 
                    training_labels, 
                    epochs=EPOCHS, 
                    validation_data = (valid_X, validation_labels), 
                    batch_size=64)

Epoch 1/3


I0000 00:00:1770733004.277608 2770890 service.cc:148] XLA service 0x7f028c013ff0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770733004.277618 2770890 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
2026-02-10 14:16:44.652888: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770733005.634073 2770890 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-02-10 14:16:47.528911: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_28167', 48 bytes spill stores, 48 bytes spill loads

2026-02-10 14:16:47.577935: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_28167', 20 bytes spil

782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step - accuracy: 0.3877 - loss: 1.7461

2026-02-10 14:20:11.902379: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3306', 188 bytes spill stores, 188 bytes spill loads



782/782 ━━━━━━━━━━━━━━━━━━━━ 226s 247ms/step - accuracy: 0.3879 - loss: 1.7454 - val_accuracy: 0.8915 - val_loss: 0.3622
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 171s 219ms/step - accuracy: 0.8673 - loss: 0.4262 - val_accuracy: 0.9385 - val_loss: 0.1986
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 172s 220ms/step - accuracy: 0.9213 - loss: 0.2537 - val_accuracy: 0.9523 - val_loss: 0.1525


In [7]:
MODEL_PATH = "../models_data/efficientnetB3/"
os.makedirs(MODEL_PATH, exist_ok=True)
model.save(MODEL_PATH+'efficientnetB3_fp32_cifar10.keras')

In [8]:
model.evaluate(valid_X, validation_labels)
model.evaluate(valid_X[:2000], validation_labels[:2000])

2026-02-10 14:26:17.421187: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3313', 4 bytes spill stores, 4 bytes spill loads

2026-02-10 14:26:17.499651: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3306', 16 bytes spill stores, 16 bytes spill loads

2026-02-10 14:26:17.571757: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3306_0', 104 bytes spill stores, 172 bytes spill loads

2026-02-10 14:26:17.593978: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3306', 188 bytes spill stores, 188 bytes spill loads



313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 30ms/step - accuracy: 0.9515 - loss: 0.1535
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9546 - loss: 0.1358


[0.1441543847322464, 0.9520000219345093]

In [9]:
def representative_dataset():
  for images in tf.data.Dataset.from_tensor_slices(train_X).batch(1).take(1000):
    yield [images]

loaded_model = keras.saving.load_model("../models_data/efficientnetB3/efficientnetB3_fp32_cifar10.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(loaded_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] 
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_quant_model = converter.convert()
with open('../models_data/efficientnetB3/efficientnetB3_uint8_cifar10.tflite', 'wb') as f:
  f.write(tflite_quant_model)

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_quant_model = converter.convert()
with open('../models_data/efficientnetB3/efficientnetB3_int8_cifar10.tflite', 'wb') as f:
  f.write(tflite_quant_model)

INFO:tensorflow:Assets written to: /tmp/tmps1n43_v9/assets


INFO:tensorflow:Assets written to: /tmp/tmps1n43_v9/assets


Saved artifact at '/tmp/tmps1n43_v9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  139649204277616: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139655426698832: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139649204324304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204326592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204324480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204324128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204326768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204331344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204330816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204334864: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/g.perlo/jacki_venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1770733637.512108 2770722 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1770733637.512120 2770722 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-02-10 14:27:17.512386: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmps1n43_v9
2026-02-10 14:27:17.523658: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-02-10 14:27:17.523670: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmps1n43_v9
I0000 00:00:1770733637.618600 2770722 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-02-10 14:27:17.638861: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-02-10 14:27:1

INFO:tensorflow:Assets written to: /tmp/tmp7wexobpj/assets


INFO:tensorflow:Assets written to: /tmp/tmp7wexobpj/assets


Saved artifact at '/tmp/tmp7wexobpj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  139649204277616: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139655426698832: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139649204324304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204326592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204324480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204324128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204326768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204331344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204330816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139649204334864: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/g.perlo/jacki_venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1770733963.119246 2770722 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1770733963.119257 2770722 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-02-10 14:32:43.119381: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp7wexobpj
2026-02-10 14:32:43.133680: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-02-10 14:32:43.133694: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp7wexobpj
2026-02-10 14:32:43.250911: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-02-10 14:32:43.920499: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tm